# How to train a new language model from scratch using Transformers and Tokenizers


The Transformer model of this Notebook is a Transformer model named ***KantaiBERT***. ***KantaiBERT*** is trained as a RoBERTa Transformer with DistilBERT architecture. The dataset was compiled with three books by Immanuel Kant downloaded from the [Gutenberg Project](https://www.gutenberg.org/). 


***KantaiBERT*** was pretrained with a small model of 84 million parameters using the same number of layers and heads as DistilBert, i.e., 6 layers, 768 hidden size,and 12 attention heads. ***KantaiBERT*** is then fine-tuned for a downstream masked Language Modeling task.

### The Hugging Face original Reference and notes:

Notebook edition (link to original of the reference blogpost [link](https://huggingface.co/blog/how-to-train)).


## Load data

In [ ]:
# Step 1: Loading the Dataset
#1.Load kant.txt using the Colab file manager
#2.Downloading the file from GitHub
!curl -L https://raw.githubusercontent.com/PacktPublishing/Transformers-for-Natural-Language-Processing/master/Chapter03/kant.txt --output "kant.txt"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 10.7M  100 10.7M    0     0  31.0M      0 --:--:-- --:--:-- --:--:-- 30.9M


In [ ]:
if  False:
    # Step 2:Installing Hugging Face Transformers
    # We won't need TensorFlow here
    !pip uninstall -y tensorflow
    # Install `transformers` from master
    !pip install git+https://github.com/huggingface/transformers
    !pip list | grep -E 'transformers|tokenizers'
    !pip install accelerate>=0.26.0
    # transformers version at notebook update --- 2.9.1
    # tokenizers version at notebook update --- 0.7.0

## Data preparation (tokenization)

In [ ]:
# Step 3: Training a Tokenizer
from pathlib import Path

from tokenizers import ByteLevelBPETokenizer

paths = [str(x) for x in Path(".").glob("**/*.txt")]    # kant.txt
# Initialize a tokenizer
tokenizer = ByteLevelBPETokenizer()

# Customize training
tokenizer.train(files=paths, vocab_size=52_000, min_frequency=2, special_tokens=[       # A token (subword) must appear at least 2 times
    "<s>",
    "<pad>",
    "</s>",
    "<unk>",        # unknown token (for anything not in vocab)
    "<mask>",
])

In [ ]:
# Step 4: Saving the files to disk
import os
token_dir = r'.\content'
if not os.path.exists(token_dir):
  os.makedirs(token_dir)
tokenizer.save_model(token_dir+ r'.\KantaiBERT')

['.\\content.\\KantaiBERT\\vocab.json', '.\\content.\\KantaiBERT\\merges.txt']

In [1]:
# Step 5 Loading the Trained Tokenizer Files 
from tokenizers.implementations import ByteLevelBPETokenizer
from tokenizers.processors import BertProcessing

tokenizer = ByteLevelBPETokenizer(
    r"./content/KantaiBERT/vocab.json",     # merged tokenized sub-string
    r"./content/KantaiBERT/merges.txt",     # indices
)

In [2]:
tokenizer.encode("The Critique of Pure Reasons is smilling.").tokens    # Ġ means space before this token

['The',
 'ĠCritique',
 'Ġof',
 'ĠPure',
 'ĠReason',
 's',
 'Ġis',
 'Ġs',
 'mill',
 'ing',
 '.']

In [3]:
encoded = tokenizer.encode("The Critique of Pure Reasons is smilling.").ids
encoded

[803, 2245, 270, 1410, 1270, 87, 300, 278, 10054, 293, 18]

In [4]:
print("Decoded:", tokenizer.decode(encoded))

Decoded: The Critique of Pure Reasons is smilling.


In [5]:
tokenizer.encode("The Critique of Pure Reasons is smilling.")

Encoding(num_tokens=11, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [6]:
# Post-processor (adding special tokens)
tokenizer._tokenizer.post_processor = BertProcessing(
    ("</s>", tokenizer.token_to_id("</s>")),
    ("<s>", tokenizer.token_to_id("<s>")),
)
tokenizer.enable_truncation(max_length=512)     # just truncate and not any padding

In [7]:
print(tokenizer.encode("The Critique of Pure Reasons is smilling to Meisam.").tokens)
print(len(tokenizer.encode("The Critique of Pure Reasons is smilling."*1000).tokens))

['<s>', 'The', 'ĠCritique', 'Ġof', 'ĠPure', 'ĠReason', 's', 'Ġis', 'Ġs', 'mill', 'ing', 'Ġto', 'ĠMe', 'is', 'am', '.', '</s>']
512


## CUDA check

In [8]:
# Step 6: Checking Resource Constraints: GPU and NVIDIA 
!nvidia-smi

Tue Sep 23 15:16:06 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.29                 Driver Version: 581.29         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090      WDDM  |   00000000:01:00.0  On |                  Off |
|  0%   34C    P8             17W /  450W |    3527MiB /  24564MiB |     28%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
# Checking that PyTorch Sees CUDAnot
import torch
torch.cuda.is_available()
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

1
NVIDIA GeForce RTX 4090


## Load model configuration

In [10]:
# Step 7: Defining the configuration of the Model
from transformers import RobertaConfig

config = RobertaConfig(
    vocab_size=52_000,
    max_position_embeddings=514,
    num_attention_heads=12,             # Each transformer layer has 12 attention heads
    num_hidden_layers=6,                # The number of transformer encoder layers
    type_vocab_size=1,
)

C:\Users\meisa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
print(config)

RobertaConfig {
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.56.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 52000
}



In [12]:
# Step 8: Re-creating the Tokenizer in Transformers
from transformers import RobertaTokenizer
tokenizer = RobertaTokenizer.from_pretrained("./content/KantaiBERT", max_length=512)

In [13]:
# Step 9: Initializing a Model From Scratch
from transformers import RobertaForMaskedLM

model = RobertaForMaskedLM(config=config)
print(model)

RobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(52000, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm)

In [14]:
print(model.num_parameters())
print(model.config.num_attention_heads)   # should be 12
print(model.config.hidden_size)          # 768
print(model.config.hidden_size // model.config.num_attention_heads)  # 64 per head


83504416
12
768
64


In [ ]:
# Step 10: Building the Dataset using Hugging Face utility class
from transformers import LineByLineTextDataset

dataset = LineByLineTextDataset(
    tokenizer=tokenizer,
    file_path="./kant.txt",
    block_size=128,             # Splits them into chunks of block_size tokens
)

C:\Users\meisa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\transformers\data\datasets\language_modeling.py:119: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


In [18]:
# Step 11: Defining a Data Collator
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)

In [24]:
data_collator

DataCollatorForLanguageModeling(tokenizer=RobertaTokenizer(name_or_path='./content/KantaiBERT', vocab_size=19296, model_max_length=1000000000000000019884624838656, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	4: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=False, special=True),
}
), mlm=True, mlm_probability=0.15

In [19]:
# Step 12: Initializing the Trainer
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./KantaiBERT",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=64,
    save_steps=10_000,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

In [20]:
# Step 13: Pre-training the Model
import time

start = time.time()

trainer.train()

end = time.time()
print(f"Elapsed time: {end - start:.4f} seconds")

Step,Training Loss
500,6.594700
1000,5.730100
1500,5.263000
2000,5.023300
2500,4.911100


Elapsed time: 97.0057 seconds


In [21]:
# Step 14: Saving the Final Model(+tokenizer + config) to disk
trainer.save_model("./KantaiBERT")

## Model prediction

In [22]:
# Step 15: Language Modeling with the FillMaskPipeline
from transformers import pipeline

fill_mask = pipeline(
    "fill-mask",
    model="./KantaiBERT",
    tokenizer="./KantaiBERT"
)

Device set to use cuda:0


In [23]:
fill_mask("Human thinking involves<mask>.")

[{'score': 0.042910851538181305,
  'token': 16,
  'token_str': ',',
  'sequence': 'Human thinking involves,.'},
 {'score': 0.01886042021214962,
  'token': 393,
  'token_str': ' reason',
  'sequence': 'Human thinking involves reason.'},
 {'score': 0.015410777181386948,
  'token': 306,
  'token_str': ' it',
  'sequence': 'Human thinking involves it.'},
 {'score': 0.013500683009624481,
  'token': 531,
  'token_str': ' experience',
  'sequence': 'Human thinking involves experience.'},
 {'score': 0.011947643011808395,
  'token': 508,
  'token_str': ' them',
  'sequence': 'Human thinking involves them.'}]